# Web Search MCP (Optional)

This optional notebook demonstrates integrating SearXNG as a web search MCP server to complement document-based research with live web results.

## Prerequisites
- SearXNG instance running (can be deployed on OpenShift)
- `SEARXNG_URL` configured in `.env`

In [ ]:
import os
import sys
import httpx
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from dotenv import load_dotenv
env_path = os.path.join('..', '.env')
load_dotenv(env_path, override=True)

_VERIFY_SSL = os.getenv("VERIFY_SSL", "true").lower() not in ("false", "0", "no")
_http_client = None if _VERIFY_SSL else httpx.Client(verify=False, timeout=httpx.Timeout(300.0))

SEARXNG_URL = os.getenv("SEARXNG_URL", "http://localhost:8888")
print(f"SearXNG URL: {SEARXNG_URL}")

## Web Search MCP Server Design

A web search MCP server would expose:
- `web_search` — Search the web via SearXNG
- `fetch_page` — Retrieve and parse a web page
- `summarize_results` — Summarize search results

In [ ]:
import httpx

async def web_search(query: str, num_results: int = 5) -> list[dict]:
    """Search the web via SearXNG."""
    params = {
        "q": query,
        "format": "json",
        "engines": "google,duckduckgo",
    }
    async with httpx.AsyncClient() as client:
        try:
            response = await client.get(f"{SEARXNG_URL}/search", params=params, timeout=15)
            response.raise_for_status()
            data = response.json()
            results = []
            for r in data.get("results", [])[:num_results]:
                results.append({
                    "title": r.get("title", ""),
                    "url": r.get("url", ""),
                    "content": r.get("content", ""),
                })
            return results
        except Exception as e:
            return [{"error": str(e)}]

# Test (uncomment when SearXNG is available)
# import asyncio
# results = asyncio.get_event_loop().run_until_complete(web_search("OpenShift AI vLLM deployment"))
# for r in results:
#     print(f"  {r.get('title', 'No title')}")
#     print(f"  {r.get('url', '')}\n")

print("Web search function defined. Uncomment test when SearXNG is available.")

## Integration with the Harness

To add web search to the research harness:
1. Deploy SearXNG on OpenShift
2. Add `web_search` to the tool layer
3. The planner can generate `{"action": "web_search", ...}` steps
4. The execute node handles web search alongside document search

This is left as an extension exercise.